# TAD Ethereum — Controller Notebook

**Reimplementation of:** Ofori-Boateng et al. (2021) — *Topological Anomaly Detection in Dynamic Multilayer Blockchain Networks* ([arXiv:2106.01806](https://arxiv.org/abs/2106.01806))

### Pipeline (runs automatically after configuration)
1. Load & filter the transaction graph to top-ranked nodes
2. For each day (and each layer), build a weighted graph → geodesic densification
3. Run clique persistent homology → Persistence Diagram (PD) per day
4. If layers are enabled, stack all layer PDs into a Stacked PD (SPD)
5. Compute Wasserstein distance W₁(PD_{t-1}, PD_t) between consecutive days
6. Extract TDA index features and compute composite indices
7. Save results — one JSON file per year, one key per layer-set

---

**Only Section 1 (Configuration) needs editing between runs.**

## 0. Setup — load function library

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(''), 'functions'))

from pathlib import Path

import warnings
warnings.filterwarnings('ignore')

from tad_ethereum_functions import run_all

print('Setup complete ✓')

## 1. Configuration

**Edit only this cell between runs.**

### `YEARS`
List of years to process. Each year gets its own output file (`run_results_V1_<YEAR>.json`).

### `LAYERS`
Dictionary of named layer-sets. Each key becomes the **run name** (entry key) inside the year's results file.  
Each value is a `LAYER_FILTERS` dict `{layer_name: callable(df) -> bool mask}`, or `None` for single-layer mode.

```python
# Example entries:

'simple_and_contracts': {
    'simple_txs':   lambda d: d['total_input_bytes'] <= 300,
    'contract_txs': lambda d: d['total_input_bytes'] >  300,
},

'by_value': {
    'low_value':  lambda d: d['tx_value'] <= d['tx_value'].median(),
    'high_value': lambda d: d['tx_value'] >  d['tx_value'].median(),
},

'single_layer': None,   # no split — whole graph as one layer
```

### Output layout
```
run_results_V1_2023.json
  └─ "simple_and_contracts"  ← run produced by LAYERS key
  └─ "by_value"

run_results_V1_2024.json
  └─ "simple_and_contracts"
  └─ "by_value"
```

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  —  edit everything in this cell
# ════════════════════════════════════════════════════════════════════════════
#
# NOTE: TDA_CFG's edge_weight_col/ranking_metric apply globally to every
# layer in one run_all() call -- to build diagrams for multiple weight
# columns (tx_count, tx_value, total_gas_fees), run this cell + the next
# once per weight, changing edge_weight_col/ranking_metric and LAYERS each
# time (tx_value only makes sense for simple_txs_ETH_only -- tx_value==0
# by definition for every contract_* filter; total_gas_fees only makes
# sense for the ETH_only layers, not contract_txs_ERC20_only -- no gas
# info for ERC20 events). run_tda_all_years.py automates exactly this
# 3-group split for the full training-window (2020-2023) rebuild; this
# cell is left as the tx_count reference config for manual/single-group
# reruns.

# ── Years to process ─────────────────────────────────────────────────────────
# Training-window only (2020-2023) -- per this project's standing
# non-circular discipline, the 2024-2025 holdout stays untouched until the
# confirmatory pipeline has run on the training-window result.
YEARS = [2020, 2021, 2022, 2023]

# ── Layer-sets ────────────────────────────────────────────────────────────────
# Key   = run name (stored as the key in each year's results JSON)
# Value = LAYER_FILTERS dict, or None for single-layer mode

    # ── Add more layer-sets here, e.g.: ──────────────────────────────────────
    # 'simple_and_contracts': {
    #     'simple_txs':   lambda d: d['total_input_bytes'] <= 300,
    #     'contract_txs': lambda d: d['total_input_bytes'] >  300,
    # },
    # 'single_layer': None,


# tx_count reference config: all 6 ETH_only layers + contract_txs_ERC20_only
# (the layer tied to the idea-21 ContractTxsAll-vs-ContractTxsEthOnly finding).
LAYERS = {
    'contract_txs_ETH_only_750': {'contract_txs_ETH_only': lambda d: (d['tx_value'] == 0) & (d['erc20']=='ETH')},
    'simple_txs_ETH_only_750': {'simple_txs_ETH_only':   lambda d: (d['tx_value'] >  0) & (d['erc20']=='ETH')},
    'contract_factory_ETH_only_750': {'contract_factory_ETH_only': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100) & (d['erc20']=='ETH')},
    'contract_nonFactory_ETH_only_750': {'contract_nonFactory_ETH_only': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']<100) & (d['erc20']=='ETH')},
    'contract_highInput_ETH_only_750': {'contract_highInput_ETH_only': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=500) & (d['erc20']=='ETH')},
    'contract_mediumInput_ETH_only_750': {'contract_mediumInput_ETH_only': lambda d: (d['tx_value'] == 0) & (d['total_input_bytes']>=100) & (d['total_input_bytes']<500) & (d['erc20']=='ETH')},
    'contract_txs_ERC20_only_750': {'contract_txs_ERC20_only': lambda d: (d['tx_value'] == 0) & (d['erc20']!='ETH')},
}



# ── TDA settings ─────────────────────────────────────────────────────────────
TDA_CFG = {
    'edge_weight_col':  'tx_count',        # column used as edge weight
    'max_nodes':        1000,              # node cap per day / layer
    'homology_maxdim':  1,                 # H0 + H1
    'distance_metric':  'wasserstein',     # 'wasserstein' or 'bottleneck'
    'similarity_metric':'norm_similarity', # 'norm_similarity' or 'i/log'
    'alpha':            9,                 # constant in norm_similarity
    'ranking_metric':   'tx_count',        # tx_count / tx_value / total_gas_fees / page_rank / ...
    'global_top':       0,                 # 0 = skip global filter
    'daily_top':        750,               # keep top-N nodes per day
}

# ── Paths ─────────────────────────────────────────────────────────────────────
# data_root / YEAR / eth_tx_value_output/weekly   ← ETH-native parquets
# data_root / YEAR / erc20_tx_value_output/weekly ← ERC20 parquets
# ranking_root / YEAR / global_top_nodes.parquet
# ranking_root / YEAR / daily/
PATH_CFG = {
    'data_root':      Path('data'),
    'ranking_root':   Path('data/ranking'),
    # New prefix, deliberately NOT run_results_V1 -- V1 is the file every
    # prior regime_detection Phase 1/2 finding was built from; keeping this
    # batch separate avoids any risk of touching it.
    'results_prefix': 'run_results_V2',          # → run_results_V2_<YEAR>.json
}

print(f'Years      : {YEARS}')
print(f'Layer-sets : {list(LAYERS.keys())}')
print(f'Total runs : {len(YEARS) * len(LAYERS)}')
print(f'Results    : {PATH_CFG["results_prefix"]}_<YEAR>.json')
print('Configuration set ✓')

<!-- --- -->
## 2. Run

Single call — no editing required.

In [ ]:
run_all(
    years    = YEARS,
    layers   = LAYERS,
    tda_cfg  = TDA_CFG,
    path_cfg = PATH_CFG,
)